# Model Analysis

Loads cross-validation metrics and out-of-sample predictions from Notebook 5. Creates leaderboards showing best fold performance and analyzes SHAP feature importance.


## Setup 

In [4]:
# Setup & paths 
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RUN_TAG   = "PROP_v1"

REPORTS   = Path("../../reports/property")
TABLES    = REPORTS / "tables"
FIGS      = REPORTS / "figures"
for p in [TABLES, FIGS]: p.mkdir(parents=True, exist_ok=True)

def read_parquet_safe(p: Path, **kw):
    if p.exists():
        return pd.read_parquet(p, **kw)
    print(f"[WARN] Missing: {p}")
    return pd.DataFrame()

def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()

# Load NB5 outputs 
met_path = TABLES / f"metrics_{RUN_TAG}_property.parquet"
if not met_path.exists():
    met_path = TABLES / f"metrics_{RUN_TAG}.parquet"
oof_all_path = TABLES / f"oof_{RUN_TAG}_ALL_property.parquet"

metrics = read_parquet_safe(met_path)
oof_all = read_parquet_safe(oof_all_path)

print(f"[LOAD] metrics rows={len(metrics):,} | cols={list(metrics.columns)}")
print(f"[LOAD] oof rows={len(oof_all):,} | cols={list(oof_all.columns)}")


[LOAD] metrics rows=48 | cols=['rmse', 'mae', 'r2', 'dir_acc', 'rank_ic', 'variant', 'task', 'target', 'fold', 'n_trn', 'n_val', 'model', 'best_iter', 'auc', 'ap', 'acc', 'f1', 'brier']
[LOAD] oof rows=99,360 | cols=['variant', 'task', 'target', 'fold', 'model', 'LA_code', 'month', 'y_true', 'y_pred', 'y_prob']


In [6]:
# Leaderboards
def choose_best(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    if (g["task"] == "reg").all():
        
        sort_cols = [c for c in ["rmse","dir_acc","r2"] if c in g.columns]
        ascending = [True] + [False]*(len(sort_cols)-1)
        g = g.sort_values(sort_cols, ascending=ascending)
    else:
        
        sort_cols = [c for c in ["auc","ap","brier"] if c in g.columns]
        ascending = [False, False, True][:len(sort_cols)]
        g = g.sort_values(sort_cols, ascending=ascending)
    return g.head(1)

if metrics.empty:
    print("[SKIP] No metrics loaded.")
else:
    best_rows = (metrics
                 .groupby(["variant","task","target"], as_index=False, group_keys=True)
                 .apply(choose_best)
                 .reset_index(drop=True))

    print(" Best fold per (variant, reg) ")
    reg_cols_show = [c for c in ["rmse","mae","r2","dir_acc","variant","task","target","fold","n_trn","n_val","best_iter"] if c in best_rows.columns]
    print(best_rows[best_rows.task=="reg"][reg_cols_show])

    print("\n Best fold per (variant, cls) ")
    cls_cols_show = [c for c in ["acc","f1","brier","auc","ap","variant","task","target","fold","n_trn","n_val","best_iter"] if c in best_rows.columns]
    print(best_rows[best_rows.task=="cls"][cls_cols_show])

    # Save leaderboards
    lb_out = TABLES / f"leaderboard_{RUN_TAG}.parquet"
    best_rows.to_parquet(lb_out, index=False)
    best_rows.to_csv(TABLES / f"leaderboard_{RUN_TAG}.csv", index=False)
    print(f"\nLeaderboard saved → {lb_out}")


 Best fold per (variant, reg) 
       rmse       mae        r2   dir_acc   variant task        target  \
2  0.064261  0.047207  0.064668  0.565608      CORE  reg  y_ret_1m_fwd   
3  0.068336  0.051716  0.112279  0.609259      CORE  reg  y_ret_3m_fwd   
6  0.067199  0.049162  0.139958  0.602207  WITHRENT  reg  y_ret_1m_fwd   
7  0.068093  0.051220  0.170610  0.612950  WITHRENT  reg  y_ret_3m_fwd   

        fold  n_trn  n_val  best_iter  
2  B_val2022  99881   3780          0  
3  A_val2020  92320   3780          0  
6  C_val2023  23825   3444          0  
7  B_val2022  20381   3444          0  

 Best fold per (variant, cls) 
        acc        f1     brier       auc        ap   variant task   target  \
0  0.581439  0.595978  0.243520  0.627359  0.694373      CORE  cls  y_up_1m   
1  0.675132  0.639460  0.232530  0.754036  0.803898      CORE  cls  y_up_3m   
4  0.598142  0.549186  0.254895  0.633679  0.564365  WITHRENT  cls  y_up_1m   
5  0.622822  0.517996  0.291330  0.764902  0.80018

/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90794/3729856144.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_best)


In [8]:
# OOF diagnostics 
if oof_all.empty:
    print("[SKIP] No OOF loaded.")
else:
    # 2A) Regression scatter & per fold summary
    reg = oof_all[oof_all["task"]=="reg"].copy()
    if not reg.empty and {"y_true","y_pred"}.issubset(reg.columns):
        for (variant, target, fold), g in reg.groupby(["variant","target","fold"]):
            fig = plt.figure(figsize=(5,4))
            plt.scatter(g["y_true"], g["y_pred"], s=6, alpha=0.5)
            lo = float(min(g["y_true"].min(), g["y_pred"].min()))
            hi = float(max(g["y_true"].max(), g["y_pred"].max()))
            plt.plot([lo,hi], [lo,hi], linestyle="--")
            plt.title(f"OOF scatter — {variant} {target} {fold}")
            plt.xlabel("y_true"); plt.ylabel("y_pred")
            savefig(FIGS / f"oof_scatter_{RUN_TAG}_{variant}_{target}_{fold}.png")

        def _reg_summary(group):
            err = group["y_pred"] - group["y_true"]
            rmse = float(np.sqrt(np.mean(np.square(err))))
            mae  = float(np.mean(np.abs(err)))
            dir_acc = float(np.mean((group["y_true"]>=0) == (group["y_pred"]>=0)))
            return pd.Series({"rmse":rmse, "mae":mae, "dir_acc":dir_acc, "n":len(group)})

        reg_summ = (reg.groupby(["variant","target","fold"], as_index=False)
                        .apply(_reg_summary))
        reg_summ.to_parquet(TABLES / f"oof_reg_summary_{RUN_TAG}.parquet", index=False)
        print("[OOF] Regression summary\n", reg_summ.head())

    # Classification: Brier & Acc per fold
    cls = oof_all[oof_all["task"]=="cls"].copy()
    if not cls.empty and {"y_true","y_prob"}.issubset(cls.columns):
        cls = cls.copy()
        cls["prob"]   = cls["y_prob"].astype(float)
        cls["y"]      = cls["y_true"].astype(int)
        cls["brier"]  = (cls["prob"] - cls["y"])**2

        def _cls_summary(group):
            brier = float(group["brier"].mean())
            acc   = float(np.mean(group["y"] == (group["prob"]>=0.5)))
            return pd.Series({"brier":brier, "acc":acc, "n":len(group)})

        cls_summ = (cls.groupby(["variant","target","fold"], as_index=False)
                        .apply(_cls_summary))
        cls_summ.to_parquet(TABLES / f"oof_cls_summary_{RUN_TAG}.parquet", index=False)
        print("\n[OOF] Classification summary\n", cls_summ.head())


[OOF] Regression summary
   variant        target       fold      rmse       mae   dir_acc       n
0    CORE  y_ret_1m_fwd  A_val2020  0.067118  0.048983  0.545961  4308.0
1    CORE  y_ret_1m_fwd  B_val2022  0.066196  0.048682  0.568942  4308.0
2    CORE  y_ret_1m_fwd  C_val2023  0.068407  0.049291  0.594011  4308.0
3    CORE  y_ret_3m_fwd  A_val2020  0.070373  0.053291  0.613742  4308.0
4    CORE  y_ret_3m_fwd  B_val2022  0.071275  0.052956  0.655292  4308.0

[OOF] Classification summary
   variant   target       fold     brier       acc       n
0    CORE  y_up_1m  A_val2020  0.340207  0.456360  4308.0
1    CORE  y_up_1m  B_val2022  0.254226  0.590297  4308.0
2    CORE  y_up_1m  C_val2023  0.254037  0.570799  4308.0
3    CORE  y_up_3m  A_val2020  0.232546  0.627205  4308.0
4    CORE  y_up_3m  B_val2022  0.238266  0.661792  4308.0


/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90794/4210003435.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_reg_summary))
/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90794/4210003435.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_cls_summary))


In [10]:
# SHAP aggregation (top-k per group) 
shap_files = list(TABLES.glob(f"shap_{RUN_TAG}_*.parquet"))
if not shap_files:
    print("[INFO] No SHAP files found.")
else:
    rows = []
    for p in shap_files:
        stem = p.stem  
        if not stem.startswith(f"shap_{RUN_TAG}_"):
            continue
        suffix = stem[len(f"shap_{RUN_TAG}_"):]  # WITHRENT_reg_y_ret_1m_fwd_A_val2020
        parts = suffix.split("_")

        if len(parts) < 4:
            print(f"[WARN] Could not parse SHAP filename: {p.name}")
            continue

        variant = parts[0]              # WITHRENT / CORE
        task    = parts[1]              # reg / cls

        # Find the fold start (A_val… / B_val… / C_val…)
        fold_idx = None
        for i in range(2, len(parts)):
            if parts[i] in ["A","B","C"] and i+1 < len(parts) and parts[i+1].startswith("val"):
                fold_idx = i
                break
        if fold_idx is None:
            print(f"[WARN] Could not find fold in: {p.name}")
            continue

        target = "_".join(parts[2:fold_idx])
        fold   = "_".join(parts[fold_idx:])

        df = pd.read_parquet(p)
        if not {"feature","abs_shap_mean"}.issubset(df.columns):
            print(f"[WARN] Missing columns in {p.name}")
            continue

        df = df[["feature","abs_shap_mean"]].copy()
        df["variant"] = variant
        df["task"]    = task
        df["target"]  = target
        df["fold"]    = fold
        rows.append(df)

    shap_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    if shap_all.empty:
        print("[INFO] SHAP files present but empty after parsing.")
    else:
        shap_mean = (shap_all
                     .groupby(["variant","task","target","feature"], as_index=False)
                     .agg(abs_shap_mean=("abs_shap_mean","mean")))
        shap_mean["rank"] = shap_mean.groupby(["variant","task","target"])["abs_shap_mean"] \
                                       .rank(ascending=False, method="first")

        topk = shap_mean[shap_mean["rank"]<=15].sort_values(["variant","task","target","rank"])
        outp = TABLES / f"shap_top15_{RUN_TAG}.parquet"
        topk.to_parquet(outp, index=False)
        print(f"SHAP top-15 saved → {outp} (rows={len(topk):,})")

        # Barplots
        for (variant, task, target), g in topk.groupby(["variant","task","target"]):
            fig = plt.figure(figsize=(6,4))
            gg = g.sort_values("abs_shap_mean", ascending=True)
            plt.barh(gg["feature"], gg["abs_shap_mean"])
            plt.title(f"SHAP top-15 — {variant} {task} {target}")
            plt.xlabel("mean(|SHAP|)")
            savefig(FIGS / f"shap_top15_{RUN_TAG}_{variant}_{task}_{target}.png")


SHAP top-15 saved → ../../reports/property/tables/shap_top15_PROP_v1.parquet (rows=120)


In [12]:
# CORE vs WITHRENT comparison
if metrics.empty:
    print("[SKIP] No metrics for variant deltas.")
else:
    keep_cols = ["variant","task","target","fold","rmse","mae","r2","dir_acc","auc","ap","acc","f1","brier","n_trn","n_val"]
    mm = metrics[[c for c in keep_cols if c in metrics.columns]].copy()

    piv = mm.pivot_table(index=["task","target","fold"],
                         columns="variant",
                         values=[c for c in ["rmse","mae","r2","dir_acc","auc","ap","acc","f1","brier"] if c in mm.columns])

    def delta(col):
        a = piv[col].get("WITHRENT")
        b = piv[col].get("CORE")
        return (a - b) if (a is not None and b is not None) else np.nan

    comp = pd.DataFrame({
        "rmse_delta"   : delta("rmse")   if "rmse"   in piv.columns.levels[0] else np.nan,
        "mae_delta"    : delta("mae")    if "mae"    in piv.columns.levels[0] else np.nan,
        "r2_delta"     : delta("r2")     if "r2"     in piv.columns.levels[0] else np.nan,
        "diracc_delta" : delta("dir_acc")if "dir_acc" in piv.columns.levels[0] else np.nan,
        "auc_delta"    : delta("auc")    if "auc"    in piv.columns.levels[0] else np.nan,
        "ap_delta"     : delta("ap")     if "ap"     in piv.columns.levels[0] else np.nan,
        "acc_delta"    : delta("acc")    if "acc"    in piv.columns.levels[0] else np.nan,
        "f1_delta"     : delta("f1")     if "f1"     in piv.columns.levels[0] else np.nan,
        "brier_delta"  : delta("brier")  if "brier"  in piv.columns.levels[0] else np.nan,
    }).reset_index()

    comp_out = TABLES / f"variant_deltas_{RUN_TAG}.parquet"
    comp.to_parquet(comp_out, index=False)
    print(f"Variant deltas saved → {comp_out}")
    print(comp.sort_values(["task","target","fold"]).fillna(np.nan).head(12))


Variant deltas saved → ../../reports/property/tables/variant_deltas_PROP_v1.parquet
   task        target       fold  rmse_delta  mae_delta  r2_delta  \
0   cls       y_up_1m  A_val2020         NaN        NaN       NaN   
1   cls       y_up_1m  B_val2022         NaN        NaN       NaN   
2   cls       y_up_1m  C_val2023         NaN        NaN       NaN   
3   cls       y_up_3m  A_val2020         NaN        NaN       NaN   
4   cls       y_up_3m  B_val2022         NaN        NaN       NaN   
5   cls       y_up_3m  C_val2023         NaN        NaN       NaN   
6   reg  y_ret_1m_fwd  A_val2020    0.001598   0.002067 -0.043590   
7   reg  y_ret_1m_fwd  B_val2022    0.004995   0.005090 -0.152121   
8   reg  y_ret_1m_fwd  C_val2023    0.002953   0.003317 -0.063657   
9   reg  y_ret_3m_fwd  A_val2020    0.000490   0.001122 -0.015234   
10  reg  y_ret_3m_fwd  B_val2022    0.000466   0.000725 -0.010113   
11  reg  y_ret_3m_fwd  C_val2023   -0.006182  -0.005276  0.136815   

    diracc_delta  

In [14]:
# Per-LA error tables
if oof_all.empty:
    print("[SKIP] No OOF for per-LA analysis.")
else:
    # REG: RMSE/MAE per LA
    reg = oof_all[oof_all["task"]=="reg"].copy()
    if not reg.empty and {"y_true","y_pred"}.issubset(reg.columns):
        reg["err"] = reg["y_pred"] - reg["y_true"]
        def _rmse(x): return float(np.sqrt(np.mean(np.square(x))))
        def _mae(x):  return float(np.mean(np.abs(x)))
        reg_la = (reg.groupby(["variant","target","fold","LA_code"], as_index=False)
                       .agg(n=("err","size"), rmse=("err",_rmse), mae=("err",_mae)))
        reg_la.to_parquet(TABLES / f"perLA_reg_{RUN_TAG}.parquet", index=False)

        ex = (reg_la.sort_values(["variant","target","fold","rmse"])
                     .groupby(["variant","target","fold"], as_index=False)
                     .apply(lambda g: pd.concat([g.head(5), g.tail(5)], ignore_index=True))
                     .reset_index(drop=True))
        print("[Per-LA] Regression extremes (few rows)")
        print(ex.head(12))

    # CLS: Brier/Acc per LA
    cls = oof_all[oof_all["task"]=="cls"].copy()
    if not cls.empty and {"y_true","y_prob"}.issubset(cls.columns):
        cls["prob"] = cls["y_prob"].astype(float)
        cls["y"]    = cls["y_true"].astype(int)
        cls["brier"] = (cls["prob"] - cls["y"])**2

        def _cls_la_summary(group):
            brier = float(group["brier"].mean())
            acc   = float(np.mean(group["y"] == (group["prob"]>=0.5)))
            return pd.Series({"brier": brier, "acc": acc, "n": len(group)})

        cls_la = (cls.groupby(["variant","target","fold","LA_code"], as_index=False)
                       .apply(_cls_la_summary)
                       .reset_index(drop=True))
        cls_la.to_parquet(TABLES / f"perLA_cls_{RUN_TAG}.parquet", index=False)

        ex = (cls_la.sort_values(["variant","target","fold","brier"])
                      .groupby(["variant","target","fold"], as_index=False)
                      .apply(lambda g: pd.concat([g.head(5), g.tail(5)], ignore_index=True))
                      .reset_index(drop=True))
        print("\n[Per-LA] Classification extremes (few rows)")
        print(ex.head(12))


/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90794/467312671.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.concat([g.head(5), g.tail(5)], ignore_index=True))


[Per-LA] Regression extremes (few rows)
   variant        target       fold    LA_code   n      rmse       mae
0     CORE  y_ret_1m_fwd  A_val2020  E06000050  12  0.010750  0.009044
1     CORE  y_ret_1m_fwd  A_val2020  E08000003  24  0.015330  0.012881
2     CORE  y_ret_1m_fwd  A_val2020  E08000012  12  0.015607  0.012292
3     CORE  y_ret_1m_fwd  A_val2020  E06000066  12  0.016223  0.013574
4     CORE  y_ret_1m_fwd  A_val2020  E06000049  12  0.016523  0.013285
5     CORE  y_ret_1m_fwd  A_val2020  E07000237  12  0.117334  0.097141
6     CORE  y_ret_1m_fwd  A_val2020  E07000135  12  0.127669  0.113885
7     CORE  y_ret_1m_fwd  A_val2020  W06000001  12  0.127672  0.100741
8     CORE  y_ret_1m_fwd  A_val2020  E09000020  24  0.139525  0.105849
9     CORE  y_ret_1m_fwd  A_val2020  E07000198  12  0.180965  0.156300
10    CORE  y_ret_1m_fwd  B_val2022  W06000011  12  0.014690  0.012707
11    CORE  y_ret_1m_fwd  B_val2022  E08000012  12  0.016827  0.013729

[Per-LA] Classification extremes (fe

/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90794/467312671.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_cls_la_summary)
/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90794/467312671.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.concat([g.head(5), g.tail(5)], ignore_index=True))


In [18]:
# Temporal breakdown (by year)
if oof_all.empty:
    print("[SKIP] No OOF for temporal analysis.")
else:
    oof_all = oof_all.copy()
    oof_all["year"] = pd.to_datetime(oof_all["month"]).dt.year

    # Regression by year
    reg = oof_all[oof_all["task"]=="reg"].copy()
    if not reg.empty and {"y_true","y_pred"}.issubset(reg.columns):
        reg["err"] = reg["y_pred"] - reg["y_true"]
        def _rmse(x): return float(np.sqrt(np.mean(np.square(x))))
        def _mae(x):  return float(np.mean(np.abs(x)))
        reg_y = (reg.groupby(["variant","target","fold","year"], as_index=False)
                    .agg(rmse=("err",_rmse), mae=("err",_mae), n=("err","size")))
        reg_y.to_parquet(TABLES / f"time_reg_by_year_{RUN_TAG}.parquet", index=False)

        for (variant, target, fold), g in reg_y.groupby(["variant","target","fold"]):
            fig = plt.figure(figsize=(5,3))
            plt.plot(g["year"], g["rmse"], marker="o")
            plt.title(f"Reg OOF RMSE by year — {variant} {target} {fold}")
            plt.xlabel("year"); plt.ylabel("RMSE")
            savefig(FIGS / f"time_reg_rmse_{RUN_TAG}_{variant}_{target}_{fold}.png")

    # Classification by year
    cls = oof_all[oof_all["task"]=="cls"].copy()
    if not cls.empty and {"y_true","y_prob"}.issubset(cls.columns):
        cls["prob"] = cls["y_prob"].astype(float)
        cls["y"]    = cls["y_true"].astype(int)
        cls["brier"] = (cls["prob"] - cls["y"])**2
        cls_y = (cls.groupby(["variant","target","fold","year"], as_index=False)
                    .agg(brier=("brier","mean"), n=("brier","size")))
        cls_y.to_parquet(TABLES / f"time_cls_by_year_{RUN_TAG}.parquet", index=False)

        for (variant, target, fold), g in cls_y.groupby(["variant","target","fold"]):
            fig = plt.figure(figsize=(5,3))
            plt.plot(g["year"], g["brier"], marker="o")
            plt.title(f"Cls OOF Brier by year — {variant} {target} {fold}")
            plt.xlabel("year"); plt.ylabel("Brier")
            savefig(FIGS / f"time_cls_brier_{RUN_TAG}_{variant}_{target}_{fold}.png")
